# LeetCode 351 — Android Unlock Patterns bằng Python

Link đề: https://leetcode.com/problems/android-unlock-patterns/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming (quy hoạch động), Backtracking (quay lui), Bitmask (mặt nạ bit)

Skeleton LeetCode:

```python
class Solution:
    def numberOfPatterns(self, m: int, n: int) -> int:
        ...
```

Ghi chú: đây là bài **Premium**; đề được viết theo hiểu biết về bài (ví dụ và con số kỳ vọng lấy theo mô tả phổ biến của bài), nên **đối chiếu lại với đề gốc nếu bạn có tài khoản Premium**. Mọi đáp án trong notebook đã được kiểm chứng bằng code (vét cạn mọi hoán vị của 9 điểm với kiểm tra hình học độc lập).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trên lưới `3 × 3` gồm 9 điểm (đánh số `1..9`), đếm số **mẫu mở khóa** (dãy điểm) có độ dài **từ `m` đến `n`**, trong đó các điểm **khác nhau** và **không được "nhảy" qua tâm một điểm chưa được chọn**.
2. Nắm các công cụ Python: tọa độ `divmod(d - 1, 3)`, quay lui (chọn → đệ quy → bỏ chọn), danh sách `visited`, **bitmask** (`1 << i`, `mask | bit`, `mask >> i & 1`, đếm bit bằng `bin(mask).count("1")`), `itertools.permutations`.
3. Đi từ **sinh mọi hoán vị rồi lọc** → **quay lui có bảng "điểm nằm giữa"** → **quay lui + đối xứng (×4, ×4, ×1)** → **DP bitmask `dp[mask][last]`** → **bảng kết quả dựng sẵn**.
4. Hiểu mấu chốt: ràng buộc chỉ phụ thuộc vào **tập điểm đã đi** và **điểm hiện tại** (không cần nhớ thứ tự), nên trạng thái `(mask, last)` đủ để đếm; và **"điểm nằm giữa"** của hai điểm chỉ tồn tại với đúng **16** cặp có thứ tự.
5. Tránh các lỗi kinh điển: bỏ qua luật nhảy qua điểm, **quên bỏ đánh dấu khi quay lui**, bảng nhảy không đối xứng, và sai mốc độ dài (`> m` thay vì `>= m`).

## 2. Đề bài & ví dụ

Màn hình khóa của Android có lưới `3 × 3` gồm các điểm, đánh số như sau:

```text
1 2 3
4 5 6
7 8 9
```

Người dùng đặt **mẫu mở khóa** bằng cách nối các điểm theo một **dãy** `k` điểm. Dãy là **hợp lệ** nếu **cả hai** điều kiện đúng:

1. Mọi điểm trong dãy **khác nhau**.
2. Nếu đoạn thẳng nối hai điểm **liên tiếp** đi qua **tâm của một điểm khác** thì điểm đó **phải đã xuất hiện trước đó** trong dãy. (Không được "nhảy xuyên" qua điểm chưa chọn.)
   - Ví dụ nối `2` với `9` mà chưa có `5` hoặc `6` vẫn **hợp lệ**, vì đoạn từ `2` tới `9` không đi qua tâm của `5` hay `6`.
   - Nối `1` với `3` mà chưa có `2` là **không hợp lệ**, vì đoạn đi qua tâm của `2`.

Ví dụ các dãy: `[4,1,3,6]` **không hợp lệ** (đoạn `1 → 3` qua `2` chưa chọn); `[4,1,9,2]` **không hợp lệ** (đoạn `1 → 9` qua `5` chưa chọn); `[2,4,1,3,6]` **hợp lệ** (`2` đã có khi nối `1 → 3`); `[6,5,4,1,9,2]` **hợp lệ** (`5` đã có khi nối `1 → 9`).

Cho hai số nguyên `m` và `n`, trả về **số mẫu mở khóa hợp lệ, khác nhau** gồm **ít nhất `m`** và **nhiều nhất `n`** điểm. Hai mẫu **khác nhau** nếu **có điểm ở mẫu này mà không ở mẫu kia, hoặc thứ tự các điểm khác nhau**.

**Ví dụ 1:** `m = 1, n = 1` ⇒ `9` (mỗi điểm là một mẫu độ dài 1).

**Ví dụ 2:** `m = 1, n = 2` ⇒ `65`.

**Ràng buộc:** `1 <= m, n <= 9`.

In [1]:
examples = [
    (1, 1, 9),
    (1, 2, 65),
]
for m, n, expected in examples:
    print("m =", m, "n =", n, "-> kỳ vọng", expected)

grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
for row in grid:
    print(*row)

m = 1 n = 1 -> kỳ vọng 9
m = 1 n = 2 -> kỳ vọng 65
1 2 3
4 5 6
7 8 9


## 3. Kiến thức nền

### 3.1. Tọa độ của điểm trên lưới `3 × 3`

Điểm `d` (từ `1` đến `9`) nằm ở hàng `r = (d - 1) // 3` và cột `c = (d - 1) % 3`; `divmod(d - 1, 3)` cho cả hai. Ví dụ `5` là `(1, 1)` (chính giữa), `9` là `(2, 2)`. Ngược lại `d = r * 3 + c + 1`.

In [2]:
for d in range(1, 10):
    r, c = divmod(d - 1, 3)
    assert r * 3 + c + 1 == d
    print(d, "->", (r, c), end="   ")
    if d % 3 == 0:
        print()

1 -> (0, 0)   2 -> (0, 1)   3 -> (0, 2)   
4 -> (1, 0)   5 -> (1, 1)   6 -> (1, 2)   
7 -> (2, 0)   8 -> (2, 1)   9 -> (2, 2)   


### 3.2. Quay lui (backtracking): chọn → đệ quy → bỏ chọn

**Quay lui** xây dãy từng điểm một. Tại mỗi bước: (1) **chọn** điểm tiếp theo hợp lệ, đánh dấu đã dùng; (2) **đệ quy** để xây tiếp; (3) khi quay về, **bỏ đánh dấu** để nhánh khác dùng lại điểm đó. Ví dụ nhỏ: đếm các dãy gồm các số khác nhau từ `{1, 2, 3}` có độ dài `1` hoặc `2`.

In [3]:
def count_sequences(max_len):
    used = [False] * 4                     # used[1..3]
    def dfs(length):
        total = 1 if length >= 1 else 0    # dãy hiện tại (đã có >= 1 phần tử) được tính
        if length == max_len:
            return total
        for x in range(1, 4):
            if not used[x]:
                used[x] = True             # chọn
                total += dfs(length + 1)   # đệ quy
                used[x] = False            # bỏ chọn
        return total
    return dfs(0)


print(count_sequences(1), count_sequences(2), count_sequences(3))   # 3, 3 + 6 = 9, 3 + 6 + 6 = 15

3 9 15


### 3.3. Tập điểm đã đi: `visited`, `set` và bitmask

Có ba cách ghi nhớ "điểm nào đã đi": danh sách `visited[d] = True/False`, `set`, hoặc **một số nguyên** mà **bit `i` bằng `1`** nếu điểm `i` đã đi.

- `1 << i`: số chỉ có bit `i` bật; `mask | (1 << i)`: thêm điểm `i`.
- `mask >> i & 1`: bằng `1` nếu điểm `i` đã có trong `mask` (nhớ `>>` thực hiện trước `&`).
- `bin(mask).count("1")`: số điểm đã đi (số bit `1`).
- `mask` có thể là chỉ số của bảng DP: với 9 điểm có `2⁹ = 512` mask.

In [4]:
mask = 0
for i in [0, 4, 8]:                        # điểm 1, 5, 9 (chỉ số 0, 4, 8)
    mask |= 1 << i
print(mask, bin(mask))                     # 273 0b100010001
print([i for i in range(9) if mask >> i & 1])      # [0, 4, 8]
print(bin(mask).count("1"), 1 << 9)        # 3 512

273 0b100010001
[0, 4, 8]
3 512


### 3.4. Bảng "điểm nằm giữa": quy tắc trung điểm

Hai điểm `a ≠ b` có **một điểm khác nằm đúng giữa** đoạn nối chúng khi và chỉ khi **tổng hàng chẵn và tổng cột chẵn**: `(ra + rb) % 2 == 0` và `(ca + cb) % 2 == 0`. Khi đó điểm giữa có tọa độ `((ra + rb) / 2, (ca + cb) / 2)`. (Lý do: hai điểm cùng "tính chẵn lẻ" theo cả hàng lẫn cột thì trung điểm là điểm lưới.) Chỉ có **16** cặp có thứ tự như vậy: `1↔3`, `1↔7`, `1↔9`, `2↔8`, `3↔7`, `3↔9`, `4↔6`, `7↔9` (mỗi cặp hai chiều). Các cặp "kiểu mã" (như `1 → 6`) và các cặp kề nhau **không** có điểm giữa.

In [5]:
skip = [[0] * 10 for _ in range(10)]        # skip[a][b] = điểm nằm giữa a và b, 0 nếu không có
for a in range(1, 10):
    for b in range(1, 10):
        ra, ca = divmod(a - 1, 3)
        rb, cb = divmod(b - 1, 3)
        if a != b and (ra + rb) % 2 == 0 and (ca + cb) % 2 == 0:
            skip[a][b] = (ra + rb) // 2 * 3 + (ca + cb) // 2 + 1

pairs = [(a, b, skip[a][b]) for a in range(1, 10) for b in range(1, 10) if skip[a][b]]
print(len(pairs))                                       # 16
print([p for p in pairs if p[0] < p[1]])                # (1,3,2) (1,7,4) (1,9,5) (2,8,5) (3,7,5) (3,9,6) (4,6,5) (7,9,8)

16
[(1, 3, 2), (1, 7, 4), (1, 9, 5), (2, 8, 5), (3, 7, 5), (3, 9, 6), (4, 6, 5), (7, 9, 8)]


### 3.5. `itertools.permutations` và thuộc tính đối xứng

`itertools.permutations(iterable, k)` sinh mọi **dãy `k` phần tử khác nhau** (có thứ tự). Với 9 điểm có `9! / (9 - k)!` dãy độ dài `k` (ví dụ `72` dãy độ dài `2`). Đó là "kho" để vét cạn.

**Đối xứng.** Xoay hoặc lật lưới không làm đổi luật; do đó **4 góc** (`1, 3, 7, 9`) có số mẫu bằng nhau, **4 cạnh** (`2, 4, 6, 8`) bằng nhau, còn **tâm** `5` đứng riêng. Ta sẽ tận dụng ở Cách 3.

In [6]:
import itertools
import math

print(len(list(itertools.permutations(range(1, 4), 2))), math.perm(9, 2))      # 6 72
print([math.perm(9, k) for k in range(1, 10)], sum(math.perm(9, k) for k in range(1, 10)))
# [9, 72, 504, 3024, 15120, 60480, 181440, 362880, 362880] 986409

6 72
[9, 72, 504, 3024, 15120, 60480, 181440, 362880, 362880] 986409


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Độ dài 1:** `9` mẫu (mỗi điểm một mẫu).

**Độ dài 2:** từ mỗi điểm đầu, đếm các điểm thứ hai được phép (chưa có điểm nào được chọn ngoài điểm đầu, nên **mọi** cặp có điểm giữa đều bị cấm):

- Từ **góc** (ví dụ `1`): có `8` điểm khác; bỏ `3` (qua `2`), `7` (qua `4`), `9` (qua `5`) ⇒ còn `5`.
- Từ **cạnh** (ví dụ `2`): có `8` điểm khác; chỉ bỏ `8` (qua `5`) ⇒ còn `7`.
- Từ **tâm** `5`: không có đoạn nào qua điểm khác ⇒ `8`.

Tổng độ dài 2: `4 × 5 + 4 × 7 + 8 = 56`. Vậy `m = 1, n = 2` cho `9 + 56 = 65` — đúng Ví dụ 2.

In [7]:
start_ok = {}
for a in range(1, 10):
    start_ok[a] = [b for b in range(1, 10) if b != a and skip[a][b] == 0]     # độ dài 2: không được có điểm giữa
    print(a, len(start_ok[a]), start_ok[a])
print("tổng độ dài 2:", sum(len(v) for v in start_ok.values()), "| m=1, n=2:", 9 + sum(len(v) for v in start_ok.values()))

1 5 [2, 4, 5, 6, 8]
2 7 [1, 3, 4, 5, 6, 7, 9]
3 5 [2, 4, 5, 6, 8]
4 7 [1, 2, 3, 5, 7, 8, 9]
5 8 [1, 2, 3, 4, 6, 7, 8, 9]
6 7 [1, 2, 3, 5, 7, 8, 9]
7 5 [2, 4, 5, 6, 8]
8 7 [1, 3, 4, 5, 6, 7, 9]
9 5 [2, 4, 5, 6, 8]
tổng độ dài 2: 56 | m=1, n=2: 65


Quy luật cho độ dài lớn hơn: một bước đi từ `cur` sang `nxt` là **hợp lệ** khi `nxt` **chưa dùng** và (**không có điểm giữa** hoặc **điểm giữa đã dùng**). Ví dụ `1 → 3` chỉ hợp lệ khi `2` đã dùng; `1 → 9` chỉ hợp lệ khi `5` đã dùng.

In [8]:
def valid_step(cur, nxt, used):
    """cur -> nxt hợp lệ nếu nxt chưa dùng và (không có điểm giữa hoặc điểm giữa đã dùng)."""
    mid = skip[cur][nxt]
    return nxt not in used and (mid == 0 or mid in used)


print(valid_step(1, 3, {1}), valid_step(1, 3, {1, 2}))        # False True
print(valid_step(1, 9, {1}), valid_step(1, 9, {1, 5}))        # False True
print(valid_step(1, 6, {1}), valid_step(2, 9, {2}))           # True True  (đoạn không qua tâm điểm nào)

False True
False True
True True


## 5. Cách 1 — Brute Force / baseline: sinh mọi hoán vị rồi lọc

**Ý tưởng.** Với mỗi độ dài `k` từ `m` đến `n`, sinh **mọi** dãy `k` điểm khác nhau bằng `itertools.permutations`, kiểm tra từng dãy có hợp lệ không và đếm. Để **kiểm tra độc lập** (không dùng quy tắc trung điểm), hàm kiểm tra dùng **hình học**: một điểm `d` nằm trên đoạn `a–b` khi nó **thẳng hàng** (tích có hướng bằng `0`) và nằm trong **hộp bao** của hai đầu mút.

In [9]:
def between_geometric(a, b):
    """Các điểm (khác a, b) có tâm nằm trên đoạn a-b: thẳng hàng và nằm trong hộp bao."""
    ra, ca = divmod(a - 1, 3)
    rb, cb = divmod(b - 1, 3)
    result = []
    for d in range(1, 10):
        if d in (a, b):
            continue
        rd, cd = divmod(d - 1, 3)
        cross = (rb - ra) * (cd - ca) - (cb - ca) * (rd - ra)          # = 0 nếu thẳng hàng
        if cross == 0 and min(ra, rb) <= rd <= max(ra, rb) and min(ca, cb) <= cd <= max(ca, cb):
            result.append(d)
    return result


BETWEEN = {(a, b): between_geometric(a, b) for a in range(1, 10) for b in range(1, 10) if a != b}


def is_valid_pattern(seq):
    if len(set(seq)) != len(seq):
        return False
    seen = set()
    for i, d in enumerate(seq):
        if i > 0:
            for mid in BETWEEN[(seq[i - 1], d)]:
                if mid not in seen:                                    # nhảy qua điểm chưa chọn
                    return False
        seen.add(d)
    return True


print([is_valid_pattern(s) for s in ([4, 1, 3, 6], [4, 1, 9, 2], [2, 4, 1, 3, 6], [6, 5, 4, 1, 9, 2])])
# [False, False, True, True]  đúng như đề

[False, False, True, True]


In [10]:
def number_of_patterns_bruteforce(m: int, n: int) -> int:
    total = 0
    for k in range(m, n + 1):
        for seq in itertools.permutations(range(1, 10), k):            # mọi dãy k điểm khác nhau
            if is_valid_pattern(seq):
                total += 1
    return total


for m, n, expected in examples:
    print(m, n, number_of_patterns_bruteforce(m, n), "(kỳ vọng", expected, ")")

1 1 9 (kỳ vọng 9 )
1 2 65 (kỳ vọng 65 )


**Độ phức tạp (Cách 1).** Số dãy cần xét là `Σ 9!/(9-k)!` (với `n = 9` là `986 409`), mỗi dãy kiểm tra `O(k)` ⇒ cỡ **`O(9! · 9)`**, Space `O(k)`. Chạy được (vài giây), nhưng **phí**: ta kiểm tra cả những dãy sai từ **đầu** (ví dụ mọi dãy bắt đầu bằng `1, 3` đều sai, nhưng vẫn sinh hết `7!` đuôi của chúng).

**Nút thắt:** lẽ ra phải **dừng ngay** khi một bước đi không hợp lệ thay vì sinh đầy đủ rồi mới lọc. Ta cần xây dãy **từng điểm một** và **cắt cành** sớm.

## 6. Ý tưởng thuật toán / bước đột phá

**Quay lui có cắt cành.** Duy trì: điểm hiện tại `cur`, các điểm đã đi, độ dài hiện tại `len`. Mỗi lần thử điểm `nxt`:

- bỏ nếu `nxt` đã đi;
- bỏ nếu `skip[cur][nxt]` là một điểm **chưa đi**;
- ngược lại đệ quy. Mỗi dãy hợp lệ có độ dài trong `[m, n]` được đếm **một lần** tại đúng nút của nó.

Vì mọi tiền tố của một dãy hợp lệ cũng hợp lệ nên cắt cành không bỏ sót.

**Hai nhận xét làm nhanh hơn nữa.**

1. **Đối xứng:** số mẫu bắt đầu từ mọi góc bằng nhau, từ mọi cạnh bằng nhau ⇒ chỉ tính từ `1`, `2`, `5` rồi nhân `4`, `4`, `1`.
2. **Chỉ nhớ `(mask, last)`:** hợp lệ của bước tiếp theo chỉ phụ thuộc **tập điểm đã đi** (`mask`) và **điểm cuối** (`last`), **không** phụ thuộc thứ tự đã đi. Vậy có thể **gộp** các dãy cùng `(mask, last)` và dùng DP (Cách 4).

In [11]:
# Thấy việc cắt cành: đếm số "nút" quay lui so với số hoán vị vét cạn cho độ dài <= 4
def count_nodes(max_len):
    visited = [False] * 10
    nodes = 0
    def dfs(cur, length):
        nonlocal nodes
        nodes += 1
        if length == max_len:
            return
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                dfs(nxt, length + 1)
        visited[cur] = False
    for start in range(1, 10):
        dfs(start, 1)
    return nodes


print("nút quay lui:", count_nodes(4), "| hoán vị vét cạn:", sum(math.perm(9, k) for k in range(1, 5)))

nút quay lui: 2009 | hoán vị vét cạn: 3609


## 7. Các cách tiếp theo

### Cách 2 — Quay lui (DFS) với `visited` và bảng `skip`

**Giải thích từng đoạn.**

1. `skip[a][b]`: điểm nằm giữa `a` và `b` (`0` nếu không có) — dựng bằng quy tắc trung điểm (mục 3.4).
2. `dfs(cur, length)`: đã có `length` điểm, điểm cuối là `cur`. Nếu `length >= m` thì dãy hiện tại hợp lệ ⇒ cộng `1`. Nếu `length == n` thì dừng.
3. Đánh dấu `visited[cur] = True`; thử mọi `nxt` chưa đi mà `skip[cur][nxt] == 0` hoặc `visited[skip[cur][nxt]]`.
4. **Bỏ đánh dấu** `visited[cur] = False` trước khi trả về.
5. Kết quả: tổng `dfs(start, 1)` với `start = 1..9`.

In [12]:
def number_of_patterns_dfs(m: int, n: int) -> int:
    visited = [False] * 10

    def dfs(cur: int, length: int) -> int:
        total = 1 if length >= m else 0            # dãy hiện tại hợp lệ và đủ dài
        if length == n:
            return total
        visited[cur] = True                        # chọn cur
        for nxt in range(1, 10):
            if visited[nxt]:
                continue
            mid = skip[cur][nxt]
            if mid == 0 or visited[mid]:           # không có điểm giữa, hoặc điểm giữa đã đi
                total += dfs(nxt, length + 1)
        visited[cur] = False                       # bỏ chọn (quay lui)
        return total

    return sum(dfs(start, 1) for start in range(1, 10))


for m, n, expected in examples:
    print(m, n, number_of_patterns_dfs(m, n), "(kỳ vọng", expected, ")")

1 1 9 (kỳ vọng 9 )
1 2 65 (kỳ vọng 65 )


In [13]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionDFS:
    def numberOfPatterns(self, m: int, n: int) -> int:
        skip = [[0] * 10 for _ in range(10)]
        for a in range(1, 10):
            for b in range(1, 10):
                ra, ca = divmod(a - 1, 3)
                rb, cb = divmod(b - 1, 3)
                if a != b and (ra + rb) % 2 == 0 and (ca + cb) % 2 == 0:
                    skip[a][b] = (ra + rb) // 2 * 3 + (ca + cb) // 2 + 1
        visited = [False] * 10

        def dfs(cur: int, length: int) -> int:
            total = 1 if length >= m else 0
            if length == n:
                return total
            visited[cur] = True
            for nxt in range(1, 10):
                if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                    total += dfs(nxt, length + 1)
            visited[cur] = False
            return total

        return sum(dfs(start, 1) for start in range(1, 10))


print(SolutionDFS().numberOfPatterns(1, 2))      # 65

65


**Độ phức tạp Cách 2:** số nút quay lui bằng số mẫu hợp lệ độ dài `<= n` (`389 497` khi `n = 9`), mỗi nút lặp `9` lựa chọn ⇒ **Time `O(9 · P)`** với `P` là số mẫu (với `n = 9` cỡ `3,5 · 10⁶` phép), **Space `O(n)`** cho đệ quy.

### Cách 3 — Quay lui + đối xứng (`4 × góc + 4 × cạnh + 1 × tâm`)

**Ý tưởng.** Xoay/lật lưới ánh xạ mẫu hợp lệ thành mẫu hợp lệ nên số mẫu bắt đầu từ `1, 3, 7, 9` bằng nhau; từ `2, 4, 6, 8` bằng nhau. Do đó chỉ chạy `dfs` từ `1`, `2`, `5` và lấy `4 · dfs(1) + 4 · dfs(2) + dfs(5)`. Công việc giảm còn khoảng `3/9` (so với 9 điểm khởi đầu).

In [14]:
def number_of_patterns_symmetry(m: int, n: int) -> int:
    visited = [False] * 10

    def dfs(cur: int, length: int) -> int:
        total = 1 if length >= m else 0
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total

    return 4 * dfs(1, 1) + 4 * dfs(2, 1) + dfs(5, 1)      # góc, cạnh, tâm


for m, n, expected in examples:
    print(m, n, number_of_patterns_symmetry(m, n), "(kỳ vọng", expected, ")")

1 1 9 (kỳ vọng 9 )
1 2 65 (kỳ vọng 65 )


In [15]:
# Code nộp LeetCode cho Cách 3
class SolutionSymmetry:
    def numberOfPatterns(self, m: int, n: int) -> int:
        skip = [[0] * 10 for _ in range(10)]
        for a in range(1, 10):
            for b in range(1, 10):
                ra, ca = divmod(a - 1, 3)
                rb, cb = divmod(b - 1, 3)
                if a != b and (ra + rb) % 2 == 0 and (ca + cb) % 2 == 0:
                    skip[a][b] = (ra + rb) // 2 * 3 + (ca + cb) // 2 + 1
        visited = [False] * 10

        def dfs(cur: int, length: int) -> int:
            total = 1 if length >= m else 0
            if length == n:
                return total
            visited[cur] = True
            for nxt in range(1, 10):
                if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                    total += dfs(nxt, length + 1)
            visited[cur] = False
            return total

        return 4 * dfs(1, 1) + 4 * dfs(2, 1) + dfs(5, 1)


print(SolutionSymmetry().numberOfPatterns(1, 2))   # 65

65


**Độ phức tạp Cách 3:** như Cách 2 nhưng chỉ `3` điểm xuất phát ⇒ cỡ `1/3` thời gian, **Space `O(n)`**.

### Cách 4 — DP bitmask: `dp[mask][last]`

**Ý tưởng.** Gọi `dp[mask][last]` là **số dãy hợp lệ** đi qua **đúng tập điểm `mask`** và **kết thúc tại `last`**. Khởi tạo `dp[1 << i][i] = 1`. Duyệt `mask` **tăng dần** (vì thêm điểm làm `mask` tăng); với mỗi `(mask, last)` có `dp > 0`, thử `nxt` chưa trong `mask` mà `skip` cho phép (điểm giữa không có hoặc đã có trong `mask`), cộng `dp[mask][last]` vào `dp[mask | 1 << nxt][nxt]`. Số dãy độ dài `k` là tổng `dp[mask][last]` với `popcount(mask) = k`. Đáp án là tổng với `m <= popcount <= n`.

**Giải thích từng đoạn.**

1. `skip[a][b]` (chỉ số `0..8`, `-1` nếu không có điểm giữa).
2. `dp = [[0] * 9 for _ in range(512)]`.
3. Với mỗi `mask` tăng dần và mỗi `last` trong `mask`: nếu độ dài `popcount(mask)` thuộc `[m, n]` thì cộng `ways` vào đáp án.
4. Mở rộng sang `nxt` hợp lệ nếu độ dài còn `< n`.

In [16]:
# bảng điểm giữa theo chỉ số 0..8 (dùng cho bitmask): -1 nếu không có
skip0 = [[-1] * 9 for _ in range(9)]
for a in range(9):
    for b in range(9):
        ra, ca, rb, cb = a // 3, a % 3, b // 3, b % 3
        if a != b and (ra + rb) % 2 == 0 and (ca + cb) % 2 == 0:
            skip0[a][b] = (ra + rb) // 2 * 3 + (ca + cb) // 2
assert all(skip0[a - 1][b - 1] == (skip[a][b] - 1 if skip[a][b] else -1) for a in range(1, 10) for b in range(1, 10))


def number_of_patterns_bitmask(m: int, n: int) -> int:
    dp = [[0] * 9 for _ in range(1 << 9)]
    for i in range(9):
        dp[1 << i][i] = 1                                   # mẫu độ dài 1
    total = 0
    for mask in range(1, 1 << 9):
        length = bin(mask).count("1")
        for last in range(9):
            ways = dp[mask][last]
            if ways == 0:
                continue
            if m <= length <= n:
                total += ways
            if length == n:
                continue
            for nxt in range(9):
                if mask >> nxt & 1:
                    continue
                mid = skip0[last][nxt]
                if mid != -1 and not (mask >> mid & 1):
                    continue                                 # nhảy qua điểm chưa đi
                dp[mask | 1 << nxt][nxt] += ways
    return total


for m, n, expected in examples:
    print(m, n, number_of_patterns_bitmask(m, n), "(kỳ vọng", expected, ")")

1 1 9 (kỳ vọng 9 )
1 2 65 (kỳ vọng 65 )


In [17]:
# Code nộp LeetCode cho Cách 4
class SolutionBitmask:
    def numberOfPatterns(self, m: int, n: int) -> int:
        skip = [[-1] * 9 for _ in range(9)]
        for a in range(9):
            for b in range(9):
                ra, ca, rb, cb = a // 3, a % 3, b // 3, b % 3
                if a != b and (ra + rb) % 2 == 0 and (ca + cb) % 2 == 0:
                    skip[a][b] = (ra + rb) // 2 * 3 + (ca + cb) // 2
        dp = [[0] * 9 for _ in range(1 << 9)]
        for i in range(9):
            dp[1 << i][i] = 1
        total = 0
        for mask in range(1, 1 << 9):
            length = bin(mask).count("1")
            for last in range(9):
                ways = dp[mask][last]
                if ways == 0:
                    continue
                if m <= length <= n:
                    total += ways
                if length == n:
                    continue
                for nxt in range(9):
                    if mask >> nxt & 1:
                        continue
                    mid = skip[last][nxt]
                    if mid != -1 and not (mask >> mid & 1):
                        continue
                    dp[mask | 1 << nxt][nxt] += ways
        return total


print(SolutionBitmask().numberOfPatterns(1, 2))   # 65

65


**Độ phức tạp Cách 4:** `2⁹ = 512` mask × `9` điểm cuối × `9` điểm kế ⇒ **Time `O(2^9 · 9 · 9)`** (khoảng `41 000` phép), **Space `O(2^9 · 9)`**. Tổng quát với `N` điểm: `O(2^N · N²)`. Nhanh hơn quay lui rất nhiều vì **gộp** các dãy cùng `(mask, last)`.

### Cách 5 — Bảng kết quả dựng sẵn (vì chỉ có 45 cặp `(m, n)`)

**Ý tưởng.** Đầu vào chỉ là `m, n ∈ [1, 9]`, tức rất ít khả năng. Số mẫu theo **từng độ dài** là hằng số đã biết; đáp án là **tổng đoạn `[m, n]`**. Ta lấy bảng này từ Cách 4 (hoặc chạy một lần) rồi **dán** vào code. Đây là "hằng số hóa": hợp lệ để nộp bài, nhưng muốn **hiểu** thì cần các cách trên.

In [18]:
# tính bảng "số mẫu theo độ dài" bằng DP bitmask (mỗi độ dài: gọi với m = n = k)
by_length = [number_of_patterns_bitmask(k, k) for k in range(1, 10)]
print(by_length)
# [9, 56, 320, 1624, 7152, 26016, 72912, 140704, 140704]
print(sum(by_length))                      # 389497  (m = 1, n = 9)

COUNT_BY_LENGTH = [9, 56, 320, 1624, 7152, 26016, 72912, 140704, 140704]
assert by_length == COUNT_BY_LENGTH


def number_of_patterns_table(m: int, n: int) -> int:
    return sum(COUNT_BY_LENGTH[m - 1:n])    # độ dài m..n ứng với chỉ số m-1..n-1


for m, n, expected in examples:
    print(m, n, number_of_patterns_table(m, n), "(kỳ vọng", expected, ")")

[9, 56, 320, 1624, 7152, 26016, 72912, 140704, 140704]
389497
1 1 9 (kỳ vọng 9 )
1 2 65 (kỳ vọng 65 )


In [19]:
# Code nộp LeetCode cho Cách 5
class SolutionTable:
    def numberOfPatterns(self, m: int, n: int) -> int:
        count_by_length = [9, 56, 320, 1624, 7152, 26016, 72912, 140704, 140704]
        return sum(count_by_length[m - 1:n])


print(SolutionTable().numberOfPatterns(1, 2), SolutionTable().numberOfPatterns(4, 9))

65 389112


**Độ phức tạp Cách 5:** **Time `O(1)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace: (a) kiểm tra từng bước của dãy `[4,1,3,6]` và `[2,4,1,3,6]` bằng quy tắc điểm giữa; (b) quay lui từ điểm `1` với `n = 3`; (c) DP bitmask với các mask nhỏ.

In [20]:
def trace_pattern(seq):
    print("dãy", seq)
    used = set()
    for i, d in enumerate(seq):
        if i == 0:
            print(f"  bắt đầu tại {d}")
        else:
            cur = seq[i - 1]
            mid = skip[cur][d]
            if d in used:
                print(f"  {cur} -> {d}: SAI (điểm {d} đã dùng)")
                return False
            if mid != 0 and mid not in used:
                print(f"  {cur} -> {d}: SAI (đi qua điểm {mid} chưa chọn, đã chọn {sorted(used)})")
                return False
            note = f"đi qua {mid} đã chọn" if mid else "không qua điểm nào"
            print(f"  {cur} -> {d}: hợp lệ ({note})")
        used.add(d)
    print("  => dãy HỢP LỆ")
    return True


assert trace_pattern([4, 1, 3, 6]) is False
assert trace_pattern([2, 4, 1, 3, 6]) is True
assert trace_pattern([6, 5, 4, 1, 9, 2]) is True

dãy [4, 1, 3, 6]
  bắt đầu tại 4
  4 -> 1: hợp lệ (không qua điểm nào)
  1 -> 3: SAI (đi qua điểm 2 chưa chọn, đã chọn [1, 4])
dãy [2, 4, 1, 3, 6]
  bắt đầu tại 2
  2 -> 4: hợp lệ (không qua điểm nào)
  4 -> 1: hợp lệ (không qua điểm nào)
  1 -> 3: hợp lệ (đi qua 2 đã chọn)
  3 -> 6: hợp lệ (không qua điểm nào)
  => dãy HỢP LỆ
dãy [6, 5, 4, 1, 9, 2]
  bắt đầu tại 6
  6 -> 5: hợp lệ (không qua điểm nào)
  5 -> 4: hợp lệ (không qua điểm nào)
  4 -> 1: hợp lệ (không qua điểm nào)
  1 -> 9: hợp lệ (đi qua 5 đã chọn)
  9 -> 2: hợp lệ (không qua điểm nào)
  => dãy HỢP LỆ


In [21]:
def trace_dfs(start, n):
    visited = [False] * 10
    results = []

    def dfs(cur, path):
        results.append(tuple(path))
        indent = "  " * (len(path) - 1)
        print(f"{indent}dãy {path}")
        if len(path) == n:
            return
        visited[cur] = True
        for nxt in range(1, 10):
            if visited[nxt]:
                continue
            mid = skip[cur][nxt]
            if mid != 0 and not visited[mid]:
                print(f"{indent}  bỏ {nxt}: {cur}->{nxt} qua {mid} chưa đi")
                continue
            dfs(nxt, path + [nxt])
        visited[cur] = False

    dfs(start, [start])
    return results


res = trace_dfs(1, 2)                                    # từ điểm 1, độ dài tối đa 2
assert len(res) == 1 + 5                                  # chính nó + 5 điểm thứ hai hợp lệ

dãy [1]
  dãy [1, 2]
  bỏ 3: 1->3 qua 2 chưa đi
  dãy [1, 4]
  dãy [1, 5]
  dãy [1, 6]
  bỏ 7: 1->7 qua 4 chưa đi
  dãy [1, 8]
  bỏ 9: 1->9 qua 5 chưa đi


In [22]:
res3 = trace_dfs(5, 2)
assert len(res3) == 1 + 8                                 # từ tâm 5: cả 8 điểm còn lại

dãy [5]
  dãy [5, 1]
  dãy [5, 2]
  dãy [5, 3]
  dãy [5, 4]
  dãy [5, 6]
  dãy [5, 7]
  dãy [5, 8]
  dãy [5, 9]


In [23]:
def trace_bitmask_dp(show_masks):
    dp = [[0] * 9 for _ in range(1 << 9)]
    for i in range(9):
        dp[1 << i][i] = 1
    for mask in range(1, 1 << 9):
        length = bin(mask).count("1")
        for last in range(9):
            ways = dp[mask][last]
            if ways == 0 or length == 3:
                continue                                  # chỉ mở rộng tới độ dài 3 để trace gọn
            for nxt in range(9):
                if mask >> nxt & 1:
                    continue
                mid = skip0[last][nxt]
                if mid != -1 and not (mask >> mid & 1):
                    continue
                dp[mask | 1 << nxt][nxt] += ways
    for mask in show_masks:
        dots = [i + 1 for i in range(9) if mask >> i & 1]
        per_last = {i + 1: dp[mask][i] for i in range(9) if dp[mask][i]}
        print(f"mask={mask:09b} (điểm {dots}): dp theo điểm cuối = {per_last}, tổng = {sum(dp[mask])}")
    by_len = {}
    for mask in range(1, 1 << 9):
        k = bin(mask).count("1")
        if k <= 3:
            by_len[k] = by_len.get(k, 0) + sum(dp[mask])
    print("tổng số dãy theo độ dài:", by_len)
    return by_len


mask_1_3 = (1 << 0) | (1 << 2)                            # {1, 3}
mask_1_2 = (1 << 0) | (1 << 1)                            # {1, 2}
mask_1_2_3 = (1 << 0) | (1 << 1) | (1 << 2)               # {1, 2, 3}
assert trace_bitmask_dp([mask_1_3, mask_1_2, mask_1_2_3]) == {1: 9, 2: 56, 3: 320}

mask=000000101 (điểm [1, 3]): dp theo điểm cuối = {}, tổng = 0
mask=000000011 (điểm [1, 2]): dp theo điểm cuối = {1: 1, 2: 1}, tổng = 2
mask=000000111 (điểm [1, 2, 3]): dp theo điểm cuối = {1: 2, 3: 2}, tổng = 4
tổng số dãy theo độ dài: {1: 9, 2: 56, 3: 320}


Đọc kết quả trace:

| mask | các điểm | `dp` theo điểm cuối | ý nghĩa |
|---|---|---|---|
| `000000101` | `{1, 3}` | rỗng, tổng `0` | không dãy nào chỉ gồm `1, 3` (nối `1 ↔ 3` cần `2` đã có) |
| `000000011` | `{1, 2}` | `{1: 1, 2: 1}` | hai dãy `2 → 1` (kết thúc ở `1`) và `1 → 2` (kết thúc ở `2`) |
| `000000111` | `{1, 2, 3}` | `{1: 2, 3: 2}`, tổng `4` | bốn dãy `1 → 2 → 3`, `3 → 2 → 1`, `2 → 1 → 3`, `2 → 3 → 1` (hai dãy kết thúc ở `3`, hai dãy kết thúc ở `1`; không dãy nào kết thúc ở `2` vì `1 ↔ 3` cần `2` có trước) |

Các tổng theo độ dài `9, 56, 320` khớp bảng `by_length`.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 4 (DP bitmask)**: không đệ quy, `O(2⁹ · 81)` thời gian, chạy rất nhanh và dễ chứng minh. Nếu bạn thấy quay lui dễ hiểu hơn, Cách 2 hoặc Cách 3 cũng được chấp nhận (chạy tốn nhiều hơn nhưng vẫn đủ nhanh vì mọi đầu vào đều nhỏ).

In [24]:
class Solution:
    def numberOfPatterns(self, m: int, n: int) -> int:
        # skip[a][b]: chỉ số (0..8) của điểm nằm giữa a và b, hoặc -1 nếu không có
        skip = [[-1] * 9 for _ in range(9)]
        for a in range(9):
            for b in range(9):
                ra, ca, rb, cb = a // 3, a % 3, b // 3, b % 3
                if a != b and (ra + rb) % 2 == 0 and (ca + cb) % 2 == 0:
                    skip[a][b] = (ra + rb) // 2 * 3 + (ca + cb) // 2

        # dp[mask][last]: số dãy hợp lệ đi đúng qua tập điểm mask và kết thúc tại last
        dp = [[0] * 9 for _ in range(1 << 9)]
        for i in range(9):
            dp[1 << i][i] = 1

        total = 0
        for mask in range(1, 1 << 9):
            length = bin(mask).count("1")
            for last in range(9):
                ways = dp[mask][last]
                if ways == 0:
                    continue
                if m <= length <= n:
                    total += ways
                if length == n:
                    continue
                for nxt in range(9):
                    if mask >> nxt & 1:
                        continue
                    mid = skip[last][nxt]
                    if mid != -1 and not (mask >> mid & 1):
                        continue                          # nhảy qua điểm chưa đi
                    dp[mask | 1 << nxt][nxt] += ways
        return total


sol = Solution()
for m, n, expected in examples:
    print(m, n, "->", sol.numberOfPatterns(m, n), "(kỳ vọng", expected, ")")

1 1 -> 9 (kỳ vọng 9 )
1 2 -> 65 (kỳ vọng 65 )


## 10. Vì sao thuật toán đúng

**Định nghĩa hợp lệ chỉ phụ thuộc vào (tập đã đi, điểm cuối).** Một dãy `p₁, p₂, ..., p_k` hợp lệ khi các điểm khác nhau và mọi bước `p_{i-1} → p_i` thỏa: điểm giữa của hai điểm này (nếu có) thuộc `{p₁, ..., p_{i-2}}`. Điều kiện của bước `i` chỉ cần biết **tập** `{p₁, ..., p_{i-1}}` và `p_{i-1}`, **không** cần thứ tự của các điểm đi trước.

**Quy tắc trung điểm đúng.** Một điểm lưới `d` nằm trên đoạn `a–b` (trong lưới `3 × 3`) khi và chỉ khi `d` là một điểm **nguyên** trên đoạn. Trên lưới nhỏ này đoạn có `gcd(|Δr|, |Δc|) − 1` điểm nguyên bên trong; với `Δr, Δc ∈ {0, ±1, ±2}` thì giá trị đó `> 0` chỉ khi `gcd = 2`, tức `Δr, Δc` đều chẵn (`0` hoặc `±2`) và không đồng thời bằng `0` — đúng là `(ra + rb)` chẵn và `(ca + cb)` chẵn với `a ≠ b`; điểm giữa là trung điểm. Ta cũng **kiểm chứng** quy tắc này bằng hình học độc lập (`BETWEEN`) ở bộ test.

**Quay lui đúng.** Mỗi tiền tố của dãy hợp lệ là hợp lệ, nên xây từng điểm và cắt cành khi vi phạm không bỏ sót. Mỗi dãy hợp lệ tương ứng với **đúng một** đường đi trong cây quay lui nên không đếm trùng; ta cộng `1` tại nút có `length >= m`.

**Đối xứng đúng.** Các phép quay/lật của lưới `3 × 3` là song ánh trên 9 điểm giữ nguyên thẳng hàng và thứ tự trên đoạn thẳng nên giữ nguyên tính hợp lệ của dãy. Chúng đưa một góc tới mọi góc khác, một cạnh tới mọi cạnh khác, nên `dfs(1) = dfs(3) = dfs(7) = dfs(9)` và `dfs(2) = dfs(4) = dfs(6) = dfs(8)`.

**DP bitmask đúng.** Các dãy kết thúc tại `last` với tập `mask` được sinh từ các dãy kết thúc tại `prev` với tập `mask − {last}` (thêm `last` hợp lệ khi `skip` cho phép) ⇒ `dp[mask][last] = Σ_prev dp[mask − {last}][prev]` trên các `prev` cho phép. Duyệt `mask` tăng dần nên `dp[mask − {last}]` đã đầy đủ. Tổng `dp` theo `popcount` đếm mọi dãy theo độ dài.

## 11. Phân tích Time / Space Complexity

Gọi `P` là số mẫu hợp lệ có độ dài `<= n` (`P = 389 497` khi `n = 9`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị + lọc | `O(9! · 9)` (≈ `10⁶` dãy × kiểm tra) | `O(n)` |
| 2. Quay lui | `O(9 · P)` | `O(n)` |
| 3. Quay lui + đối xứng | `≈ O(3 · P)` | `O(n)` |
| 4. DP bitmask | `O(2⁹ · 9 · 9)` | `O(2⁹ · 9)` |
| 5. Bảng dựng sẵn | `O(1)` | `O(1)` |

Tổng quát với `N` điểm: DP bitmask `O(2^N · N²)` thay vì `O(N!)` của vét cạn.

## 12. Bộ test

Ta kiểm tra: (1) dãy mẫu trong đề; (2) **oracle độc lập** `BRUTE_BY_LENGTH` — vét cạn **toàn bộ** hoán vị của 9 điểm với kiểm tra hình học (không dùng quy tắc trung điểm, không dùng DP); (3) quy tắc trung điểm trùng hình học; (4) tất cả cách trên **mọi cặp `(m, n)`** và ngẫu nhiên.

In [25]:
import random
import time

# (1) các dãy ví dụ trong đề
assert [is_valid_pattern(s) for s in ([4, 1, 3, 6], [4, 1, 9, 2], [2, 4, 1, 3, 6], [6, 5, 4, 1, 9, 2])] == [False, False, True, True]
assert is_valid_pattern([2, 9]) and not is_valid_pattern([1, 3]) and is_valid_pattern([1, 2, 3])
assert not is_valid_pattern([1, 1])                         # điểm trùng
print("PASS: dãy mẫu của đề")

# (3) quy tắc trung điểm khớp hình học
for a in range(1, 10):
    for b in range(1, 10):
        if a == b:
            continue
        geo = BETWEEN[(a, b)]
        assert len(geo) <= 1, (a, b, geo)                     # trên lưới 3x3 tối đa 1 điểm giữa
        assert (geo[0] if geo else 0) == skip[a][b], (a, b, geo, skip[a][b])
print("PASS: bảng skip (quy tắc trung điểm) trùng kiểm tra hình học cho mọi cặp")

PASS: dãy mẫu của đề
PASS: bảng skip (quy tắc trung điểm) trùng kiểm tra hình học cho mọi cặp


In [26]:
# (2) Oracle độc lập: vét cạn mọi hoán vị của 9 điểm (có thể mất vài giây tùy máy)
t0 = time.perf_counter()
BRUTE_BY_LENGTH = []
for k in range(1, 10):
    BRUTE_BY_LENGTH.append(sum(1 for seq in itertools.permutations(range(1, 10), k) if is_valid_pattern(seq)))
print("vét cạn theo độ dài:", BRUTE_BY_LENGTH, f"({time.perf_counter() - t0:.1f}s)")
assert BRUTE_BY_LENGTH == COUNT_BY_LENGTH
assert sum(BRUTE_BY_LENGTH) == 389497

vét cạn theo độ dài: [9, 56, 320, 1624, 7152, 26016, 72912, 140704, 140704] (0.9s)


In [27]:
def oracle(m, n):
    return sum(BRUTE_BY_LENGTH[m - 1:n])


sol = Solution()
impls_fast = {
    "bitmask": number_of_patterns_bitmask,
    "table": number_of_patterns_table,
    "leetcode": sol.numberOfPatterns,
    "submit_bitmask": SolutionBitmask().numberOfPatterns,
    "submit_table": SolutionTable().numberOfPatterns,
}
impls_slow = {
    "dfs": number_of_patterns_dfs,
    "symmetry": number_of_patterns_symmetry,
    "submit_dfs": SolutionDFS().numberOfPatterns,
    "submit_symmetry": SolutionSymmetry().numberOfPatterns,
}

for m, n, expected in examples:
    assert oracle(m, n) == expected
    for fn in list(impls_fast.values()) + list(impls_slow.values()) + [number_of_patterns_bruteforce]:
        assert fn(m, n) == expected, (fn, m, n)
print("PASS: ví dụ đề cho mọi cách")

PASS: ví dụ đề cho mọi cách


### 12.1. Các trường hợp biên

`m = n = 1`, `m = n = 9`, `m = 1, n = 9` (tối đa), và trường hợp `m > n` (ngoài tinh thần đề nhưng ràng buộc `1 <= m, n <= 9` không cấm: không có độ dài nào thỏa nên đáp án là `0`).

In [28]:
cases = [
    (1, 1, 9), (2, 2, 56), (3, 3, 320), (4, 4, 1624),
    (9, 9, 140704), (8, 8, 140704), (8, 9, 281408),
    (1, 9, 389497),
    (4, 9, 389497 - 9 - 56 - 320),
]
for m, n, expected in cases:
    assert oracle(m, n) == expected, (m, n)
    for key, fn in impls_fast.items():
        assert fn(m, n) == expected, (key, m, n)
print("PASS: biên cho các cách nhanh")

# m > n: không có độ dài nào => 0
for m, n in [(2, 1), (9, 1), (5, 4)]:
    assert oracle(m, n) == 0
    for key, fn in impls_fast.items():
        assert fn(m, n) == 0, (key, m, n)
    assert number_of_patterns_dfs(m, n) == 0
print("PASS: m > n cho 0")

# các cách chậm (vét cạn, quay lui) trên các biên có n nhỏ (n = 9 được đo riêng ở mục 12.4)
for m, n in [(1, 4), (3, 5), (5, 5), (2, 6)]:
    for key, fn in impls_slow.items():
        assert fn(m, n) == oracle(m, n), (key, m, n)
assert number_of_patterns_bruteforce(1, 5) == oracle(1, 5)
print("PASS: biên cho các cách chậm")

PASS: biên cho các cách nhanh
PASS: m > n cho 0
PASS: biên cho các cách chậm


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách "nghe hợp lý" nhưng sai: (a) **cho phép mọi cặp điểm** (bỏ luật nhảy qua tâm); (b) **cấm mọi bước dài** (không bao giờ cho đi qua điểm đã chọn); (c) **đếm mọi hoán vị rồi lấy trừ đi** sai khác.

In [29]:
def wrong_no_rule(m, n):
    return sum(math.perm(9, k) for k in range(m, n + 1))              # SAI: coi mọi hoán vị là hợp lệ


def wrong_never_jump(m, n):
    # SAI: không bao giờ cho đi qua điểm giữa, kể cả khi điểm giữa đã chọn
    visited = [False] * 10
    def dfs(cur, length):
        total = 1 if length >= m else 0
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and skip[cur][nxt] == 0:
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total
    return sum(dfs(s, 1) for s in range(1, 10))


for m, n in [(1, 2), (1, 3), (3, 3)]:
    print((m, n), "đúng:", oracle(m, n), "| bỏ luật:", wrong_no_rule(m, n), "| không cho nhảy:", wrong_never_jump(m, n))
assert wrong_no_rule(1, 2) != oracle(1, 2)
assert wrong_never_jump(3, 3) != oracle(3, 3)
assert wrong_never_jump(1, 2) == oracle(1, 2)      # ở độ dài <= 2 hai cách giống nhau: phản ví dụ phải có độ dài >= 3

(1, 2) đúng: 65 | bỏ luật: 81 | không cho nhảy: 65
(1, 3) đúng: 385 | bỏ luật: 585 | không cho nhảy: 369
(3, 3) đúng: 320 | bỏ luật: 504 | không cho nhảy: 304


### 12.3. Đối chiếu mọi cặp `(m, n)` và ngẫu nhiên có seed

Có đúng `81` cặp `(m, n)` (kể cả `m > n`). Cách nhanh được so với oracle trên **toàn bộ** các cặp; các cách chậm chỉ trên các cặp có `n <= 7` (để giữ thời gian chạy hợp lý), cộng một lượt ngẫu nhiên có seed.

In [30]:
for m in range(1, 10):
    for n in range(1, 10):
        exp = oracle(m, n)
        for key, fn in impls_fast.items():
            assert fn(m, n) == exp, (key, m, n)
print("PASS: cách nhanh khớp oracle trên cả 81 cặp (m, n)")

t0 = time.perf_counter()
for m in range(1, 8):
    for n in range(m, 8):
        exp = oracle(m, n)
        for key, fn in impls_slow.items():
            assert fn(m, n) == exp, (key, m, n)
print(f"PASS: quay lui (cả đối xứng) khớp oracle trên mọi cặp m <= n <= 7 ({time.perf_counter() - t0:.1f}s)")

rng = random.Random(351)
for _ in range(300):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    exp = oracle(m, n)
    assert all(fn(m, n) == exp for fn in impls_fast.values()), (m, n)
for _ in range(20):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    assert number_of_patterns_bruteforce(m, n) == oracle(m, n)
print("PASS: ngẫu nhiên có seed")

# đối xứng: số mẫu từ mọi góc bằng nhau, từ mọi cạnh bằng nhau
def dfs_from(start, m, n):
    visited = [False] * 10
    def dfs(cur, length):
        total = 1 if length >= m else 0
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total
    return dfs(start, 1)

corner = {dfs_from(s, 1, 6) for s in (1, 3, 7, 9)}
edge = {dfs_from(s, 1, 6) for s in (2, 4, 6, 8)}
assert len(corner) == 1 and len(edge) == 1
print("PASS: đối xứng góc/cạnh; góc:", corner, "cạnh:", edge, "tâm:", dfs_from(5, 1, 6))

PASS: cách nhanh khớp oracle trên cả 81 cặp (m, n)


PASS: quay lui (cả đối xứng) khớp oracle trên mọi cặp m <= n <= 7 (0.5s)


PASS: ngẫu nhiên có seed
PASS: đối xứng góc/cạnh; góc: {3391} cạnh: {3975} tâm: 5713


### 12.4. Hiệu năng

Mỗi lời giải nhanh nên chạy trong tích tắc với mọi `(m, n)`. Đo trường hợp nặng nhất `(1, 9)`.

In [31]:
for key, fn in [("bitmask", number_of_patterns_bitmask), ("symmetry", number_of_patterns_symmetry), ("dfs", number_of_patterns_dfs), ("table", number_of_patterns_table)]:
    t0 = time.perf_counter()
    assert fn(1, 9) == 389497
    print(f"{key:10s} (1, 9): {(time.perf_counter() - t0) * 1000:9.1f} ms")

bitmask    (1, 9):       1.7 ms
symmetry   (1, 9):      39.2 ms
dfs        (1, 9):     102.9 ms
table      (1, 9):       0.0 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không xét luật nhảy qua điểm (cho đi mọi cặp)

In [32]:
def sai_khong_xet_luat(m, n):
    visited = [False] * 10
    def dfs(cur, length):
        total = 1 if length >= m else 0
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt]:                       # SAI: thiếu kiểm tra điểm giữa
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total
    return sum(dfs(s, 1) for s in range(1, 10))


print("sai:", sai_khong_xet_luat(1, 2), "| đúng:", Solution().numberOfPatterns(1, 2))

sai: 81 | đúng: 65


**Vì sao sai:** nếu bỏ luật thì mọi dãy điểm khác nhau đều được tính: độ dài `2` có `72` dãy thay vì `56` (thừa `16` cặp có điểm giữa), nên `m = 1, n = 2` ra `81` thay vì `65`. **Cách sửa:** chỉ đi `cur → nxt` khi `skip[cur][nxt] == 0` hoặc điểm giữa đã `visited`.

### Lỗi 2: Quên bỏ đánh dấu khi quay lui

In [33]:
def sai_quen_bo_danh_dau(m, n):
    visited = [False] * 10
    def dfs(cur, length):
        total = 1 if length >= m else 0
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                total += dfs(nxt, length + 1)
        # SAI: thiếu visited[cur] = False khi quay lui
        return total
    return sum(dfs(s, 1) for s in range(1, 10))


print("sai:", sai_quen_bo_danh_dau(1, 2), "| đúng:", Solution().numberOfPatterns(1, 2))

sai: 37 | đúng: 65


**Vì sao sai:** điểm đã đánh dấu mà không **bỏ** khi quay về làm các nhánh sau (và các điểm xuất phát sau) coi điểm đó là "đã đi" mãi mãi, nên đếm thiếu. **Cách sửa:** luôn có cặp *chọn / bỏ chọn* đối xứng quanh lời gọi đệ quy (`visited[cur] = True ... visited[cur] = False`).

### Lỗi 3: Bảng điểm giữa không đối xứng

In [34]:
def sai_skip_mot_chieu(m, n):
    skip_one = [[0] * 10 for _ in range(10)]
    for a in range(1, 10):
        for b in range(a + 1, 10):                    # SAI: chỉ điền a < b, thiếu chiều ngược lại
            skip_one[a][b] = skip[a][b]
    visited = [False] * 10
    def dfs(cur, length):
        total = 1 if length >= m else 0
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and (skip_one[cur][nxt] == 0 or visited[skip_one[cur][nxt]]):
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total
    return sum(dfs(s, 1) for s in range(1, 10))


print("sai:", sai_skip_mot_chieu(1, 2), "| đúng:", Solution().numberOfPatterns(1, 2))

sai: 73 | đúng: 65


**Vì sao sai:** đi `3 → 1` cũng phải qua `2` như `1 → 3`. Nếu bảng chỉ điền một chiều thì các bước ngược `b → a` (với `a < b`) bị cho đi tự do. **Cách sửa:** điền `skip[a][b]` **và** `skip[b][a]` (hoặc tính bằng quy tắc trung điểm cho mọi cặp có thứ tự).

### Lỗi 4: Sai mốc độ dài (`> m` thay vì `>= m`)

In [35]:
def sai_moc_do_dai(m, n):
    visited = [False] * 10
    def dfs(cur, length):
        total = 1 if length > m else 0                 # SAI: phải là length >= m
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total
    return sum(dfs(s, 1) for s in range(1, 10))


print("sai (1,1):", sai_moc_do_dai(1, 1), "| đúng:", Solution().numberOfPatterns(1, 1))
print("sai (1,2):", sai_moc_do_dai(1, 2), "| đúng:", Solution().numberOfPatterns(1, 2))

sai (1,1): 0 | đúng: 9
sai (1,2): 56 | đúng: 65


**Vì sao sai:** đề là "**ít nhất** `m` điểm", nên dãy có **đúng** `m` điểm cũng được tính. Dùng `>` bỏ mất độ dài `m` (ví dụ `m = n = 1` cho `0` thay vì `9`). **Cách sửa:** `length >= m`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm các **dãy / hoán vị** thỏa ràng buộc cục bộ, số phần tử nhỏ (`<= 9–12`)" ⇒ **quay lui**, và nếu trạng thái chỉ cần `(tập đã dùng, phần tử cuối)` thì **DP bitmask** (tương tự **Travelling Salesman**, **Number of Ways to Wear Different Hats**).
- Khi các phần tử "giống nhau về cấu trúc" (lưới đối xứng) ⇒ nghĩ ngay tới **đối xứng** để chia nhỏ công việc.
- **Quy tắc trung điểm / gcd**: điểm nguyên trên đoạn thẳng giữa hai điểm lưới có `gcd(|Δr|, |Δc|) − 1` điểm bên trong; trên lưới `3 × 3` chỉ khi `Δ` chẵn.
- Mẫu **chọn → đệ quy → bỏ chọn** phải **đối xứng**; thiếu dòng bỏ chọn là lỗi phổ biến nhất.
- Bitmask: `mask >> i & 1` kiểm tra bit, `mask | 1 << i` bật bit, `bin(mask).count("1")` đếm bit; chỉ số `mask` duyệt **tăng dần** để DP đúng thứ tự.
- Với đầu vào rất nhỏ (45 cặp) vẫn nên **chứng minh/kiểm tra** thay vì dán hằng số bừa; hằng số lấy từ oracle đã kiểm.

## 15. Biến thể của bài

1. **Lưới `R × C` tổng quát:** điểm giữa dựa trên `gcd` của `Δr, Δc` (có thể có nhiều điểm giữa).
2. **Liệt kê** (không chỉ đếm) mọi mẫu độ dài `k`, hoặc mẫu thứ `t` theo thứ tự từ điển.
3. **Mẫu bắt đầu từ một điểm cố định / kết thúc ở một điểm cố định.**
4. **Knight Dialer (935)**, **Unique Paths III (980)**, **Number of Ways to Wear Different Hats (1434)**: cùng họ quay lui / DP bitmask.
5. **Kiểm tra một mẫu có hợp lệ không** (hàm `is_valid_pattern` ở mục 5).

In [36]:
def count_patterns_grid(rows, cols, m, n):
    """Biến thể: lưới rows x cols; các điểm nằm giữa tính bằng gcd (có thể nhiều điểm)."""
    cells = [(r, c) for r in range(rows) for c in range(cols)]
    k = len(cells)
    between = {}
    for i, (r1, c1) in enumerate(cells):
        for j, (r2, c2) in enumerate(cells):
            if i == j:
                continue
            dr, dc = r2 - r1, c2 - c1
            g = math.gcd(abs(dr), abs(dc))
            between[(i, j)] = [cells.index((r1 + t * dr // g, c1 + t * dc // g)) for t in range(1, g)]
    visited = [False] * k

    def dfs(cur, length):
        total = 1 if length >= m else 0
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(k):
            if not visited[nxt] and all(visited[mid] for mid in between[(cur, nxt)]):
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total

    return sum(dfs(s, 1) for s in range(k))


print(count_patterns_grid(3, 3, 1, 2), count_patterns_grid(3, 3, 1, 9))      # 65 389497
print(count_patterns_grid(2, 2, 1, 4), count_patterns_grid(2, 3, 1, 6))
assert count_patterns_grid(3, 3, 1, 5) == oracle(1, 5)
assert count_patterns_grid(2, 2, 1, 4) == 4 + 12 + 24 + 24            # lưới 2x2 không có điểm giữa

65 389497
64 1392


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị + lọc | sinh hết, kiểm tra từng dãy | `O(9! · 9)` | `O(n)` | oracle kiểm thử |
| 2. Quay lui | xây từng điểm, cắt cành | `O(9 · P)` | `O(n)` | dễ hiểu nhất, lưới nhỏ |
| 3. Quay lui + đối xứng | chỉ chạy từ `1, 2, 5` | `≈ O(3 · P)` | `O(n)` | tăng tốc ×3 gần như miễn phí |
| 4. DP bitmask | `dp[mask][last]` | `O(2⁹ · 81)` | `O(2⁹ · 9)` | **nên nộp**, tổng quát cho `N` điểm |
| 5. Bảng dựng sẵn | cộng đoạn hằng số | `O(1)` | `O(1)` | khi đã kiểm chứng và muốn nhanh nhất |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `divmod(4, 3)` (tọa độ điểm `5`), `(1 << 3) | (1 << 0)`, `bin(0b1011).count("1")`, `0b101 >> 2 & 1`, `Solution().numberOfPatterns(1, 1)`.

**Bài 2 (tính tay).** Đếm số mẫu **độ dài `3` bắt đầu từ điểm `5`** (gợi ý: bước 1 từ tâm đi tới mọi điểm; xét từ góc và từ cạnh, bước 2 còn bao nhiêu điểm hợp lệ khi `5` đã chọn).

**Bài 3 (điền TODO).** Hoàn thành `count_dfs_todo(m, n)` theo Cách 2 (quay lui với `visited` và `skip`).

In [37]:
def count_dfs_todo(m, n):
    # TODO 1: dựng bảng skip[a][b] (a, b từ 1..9) bằng quy tắc trung điểm
    # TODO 2: visited = [False] * 10
    # TODO 3: def dfs(cur, length): total = 1 nếu length >= m; dừng nếu length == n
    # TODO 4: đánh dấu cur; thử mọi nxt chưa đi mà skip[cur][nxt] == 0 hoặc visited[skip[cur][nxt]]
    # TODO 5: BỎ đánh dấu cur (quay lui) rồi trả về total
    # TODO 6: trả về tổng dfs(s, 1) với s = 1..9
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `count_bitmask2(m, n)` theo Cách 4 (DP bitmask `dp[mask][last]`) **mà không nhìn lại code**, rồi kiểm tra `count_bitmask2(1, 9) == 389497`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `list_patterns(k, start)` trả **danh sách** mọi mẫu hợp lệ độ dài `k` bắt đầu từ điểm `start` (dãy các số). (b) Viết `count_ending_at(k, end)`: số mẫu hợp lệ độ dài `k` **kết thúc** tại điểm `end` (dùng `dp[mask][last]` và cộng theo `last = end`).

In [38]:
def list_patterns_todo(k, start):
    # TODO: quay lui từ start, thu thập các dãy có độ dài đúng k
    pass


def count_ending_at_todo(k, end):
    # TODO: DP bitmask, cộng dp[mask][end - 1] cho mọi mask có popcount == k
    pass

# KHÔNG gọi hàm TODO ở đây.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(1, 1)`, `9`, `3`, `1`, `9`. Bài 3: `skip[a][b]` chỉ khác `0` khi `(ra + rb)` và `(ca + cb)` đều chẵn và `a != b`.
- **Mức 2 (vừa):** Bài 2: từ tâm `5` mọi điểm tiếp theo đều hợp lệ (`8` cách); với điểm thứ hai là **góc** (`4` góc), điểm thứ ba có `5` lựa chọn (không qua điểm chưa chọn, nhưng nối qua `5` đã chọn thì được); với điểm thứ hai là **cạnh** (`4` cạnh), điểm thứ ba có `7` lựa chọn. Bài 4: duyệt `mask` tăng dần và nhớ điều kiện `mid != -1 and not (mask >> mid & 1)` để bỏ qua.
- **Mức 3 (sâu):** Bài 2 kết quả `4 * 5 + 4 * 7 = 48`. Bài 5a: `dfs(cur, path)`: nếu `len(path) == k` thì `append(path)`; bài 5b: `sum(dp[mask][end - 1] for mask in range(512) if bin(mask).count("1") == k)`; kiểm tra `Σ_end count_ending_at(k, end) == COUNT_BY_LENGTH[k - 1]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [39]:
# Bài 1
assert divmod(4, 3) == (1, 1)
assert (1 << 3) | (1 << 0) == 9
assert bin(0b1011).count("1") == 3
assert 0b101 >> 2 & 1 == 1
assert Solution().numberOfPatterns(1, 1) == 9

# Bài 2: số mẫu độ dài 3 bắt đầu từ 5 = 4*5 + 4*7 = 48
def dfs_from_len(start, length_target):
    visited = [False] * 10
    def dfs(cur, length):
        if length == length_target:
            return 1
        visited[cur] = True
        total = 0
        for nxt in range(1, 10):
            if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total
    return dfs(start, 1)

assert dfs_from_len(5, 3) == 48 == 4 * 5 + 4 * 7
assert dfs_from_len(5, 2) == 8

# Bài 3
def count_dfs_todo_ans(m, n):
    skip_t = [[0] * 10 for _ in range(10)]
    for a in range(1, 10):
        for b in range(1, 10):
            ra, ca = divmod(a - 1, 3)
            rb, cb = divmod(b - 1, 3)
            if a != b and (ra + rb) % 2 == 0 and (ca + cb) % 2 == 0:
                skip_t[a][b] = (ra + rb) // 2 * 3 + (ca + cb) // 2 + 1
    visited = [False] * 10

    def dfs(cur, length):
        total = 1 if length >= m else 0
        if length == n:
            return total
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and (skip_t[cur][nxt] == 0 or visited[skip_t[cur][nxt]]):
                total += dfs(nxt, length + 1)
        visited[cur] = False
        return total

    return sum(dfs(s, 1) for s in range(1, 10))

for m, n in [(1, 1), (1, 2), (2, 4), (3, 3), (5, 6)]:
    assert count_dfs_todo_ans(m, n) == oracle(m, n)

# Bài 4
def count_bitmask2(m, n):
    dp = [[0] * 9 for _ in range(512)]
    for i in range(9):
        dp[1 << i][i] = 1
    total = 0
    for mask in range(1, 512):
        k = bin(mask).count("1")
        for last in range(9):
            w = dp[mask][last]
            if not w:
                continue
            if m <= k <= n:
                total += w
            for nxt in range(9):
                if mask >> nxt & 1:
                    continue
                mid = skip0[last][nxt]
                if mid != -1 and not (mask >> mid & 1):
                    continue
                dp[mask | 1 << nxt][nxt] += w
    return total

assert count_bitmask2(1, 9) == 389497
for m, n in [(1, 2), (4, 6), (9, 9)]:
    assert count_bitmask2(m, n) == oracle(m, n)

# Bài 5a
def list_patterns(k, start):
    visited = [False] * 10
    out = []
    def dfs(cur, path):
        if len(path) == k:
            out.append(tuple(path))
            return
        visited[cur] = True
        for nxt in range(1, 10):
            if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
                dfs(nxt, path + [nxt])
        visited[cur] = False
    dfs(start, [start])
    return out

pats = list_patterns(3, 1)
assert len(pats) == 31 and all(is_valid_pattern(p) and p[0] == 1 for p in pats)
assert len(set(pats)) == len(pats)
assert len(list_patterns(2, 2)) == 7

# Bài 5b
def count_ending_at(k, end):
    dp = [[0] * 9 for _ in range(512)]
    for i in range(9):
        dp[1 << i][i] = 1
    for mask in range(1, 512):
        for last in range(9):
            w = dp[mask][last]
            if not w:
                continue
            for nxt in range(9):
                if mask >> nxt & 1:
                    continue
                mid = skip0[last][nxt]
                if mid != -1 and not (mask >> mid & 1):
                    continue
                dp[mask | 1 << nxt][nxt] += w
    return sum(dp[mask][end - 1] for mask in range(512) if bin(mask).count("1") == k)

for k in range(1, 10):
    assert sum(count_ending_at(k, e) for e in range(1, 10)) == COUNT_BY_LENGTH[k - 1]
assert count_ending_at(2, 1) == 5 and count_ending_at(2, 2) == 7 and count_ending_at(2, 5) == 8
print("Tất cả đáp án bài tập đều pass")

Tất cả đáp án bài tập đều pass


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DP bitmask)

```python
class Solution:
    def numberOfPatterns(self, m: int, n: int) -> int:
        skip = [[-1] * 9 for _ in range(9)]
        for a in range(9):
            for b in range(9):
                ra, ca, rb, cb = a // 3, a % 3, b // 3, b % 3
                if a != b and (ra + rb) % 2 == 0 and (ca + cb) % 2 == 0:
                    skip[a][b] = (ra + rb) // 2 * 3 + (ca + cb) // 2
        dp = [[0] * 9 for _ in range(1 << 9)]
        for i in range(9):
            dp[1 << i][i] = 1
        total = 0
        for mask in range(1, 1 << 9):
            length = bin(mask).count("1")
            for last in range(9):
                ways = dp[mask][last]
                if not ways:
                    continue
                if m <= length <= n:
                    total += ways
                if length == n:
                    continue
                for nxt in range(9):
                    if mask >> nxt & 1:
                        continue
                    mid = skip[last][nxt]
                    if mid != -1 and not (mask >> mid & 1):
                        continue
                    dp[mask | 1 << nxt][nxt] += ways
        return total
```

**Template quay lui** (khung)

```python
def dfs(cur, length):
    total = 1 if length >= m else 0
    if length == n:
        return total
    visited[cur] = True
    for nxt in range(1, 10):
        if not visited[nxt] and (skip[cur][nxt] == 0 or visited[skip[cur][nxt]]):
            total += dfs(nxt, length + 1)
    visited[cur] = False
    return total
```

**Checklist**

- [ ] Mỗi bước kiểm tra: `nxt` chưa đi **và** (không có điểm giữa **hoặc** điểm giữa đã đi)?
- [ ] Bảng điểm giữa **đối xứng** (`skip[a][b]` và `skip[b][a]`), đúng `16` cặp có thứ tự?
- [ ] Quay lui có đủ cặp **chọn / bỏ chọn** (`visited[cur] = True ... = False`)?
- [ ] Đếm khi `length >= m` (không phải `> m`) và dừng khi `length == n`?
- [ ] DP bitmask duyệt `mask` **tăng dần**, khởi tạo `dp[1 << i][i] = 1`?
- [ ] Đã thử `(1, 1) -> 9`, `(1, 2) -> 65`, `(1, 9) -> 389497`, và `m > n` cho `0`?
- [ ] Skeleton đúng `class Solution` / `def numberOfPatterns(self, m: int, n: int) -> int`?